In [3]:
"""
Scraper v3 — eldiario.es opinión vía SITEMAPS MENSUALES.

Esta es la solución correcta. eldiario.es publica sitemaps XML mensuales
con TODAS las URLs publicadas ese mes y su fecha exacta:

  https://www.eldiario.es/sitemap_contents_<YYYY>_<MM>_25b87_001.xml

Ventaja crítica: iteramos los meses hacia atrás desde hoy y sacamos URLs
ordenadas cronológicamente. No hay paginación rota, no hay listados que
mienten. Es el índice oficial del propio periódico.

Estrategia:
  1. Para cada mes (de más reciente a más antiguo), descargar el sitemap.
  2. Parsear con regex (más rápido y robusto que XML strict cuando los
     ficheros pesan varios MB).
  3. Filtrar solo URLs de opinión.
  4. Para cada URL, descargar artículo, comprobar paywall, extraer texto.
  5. Volcar a CSV con checkpoint cada 25 artículos.

Detector de paywall: solo descarta con marcadores explícitos.

Requisitos: pip install requests beautifulsoup4 lxml tqdm
Uso:        python scrape_eldiario_sitemap.py
"""

from __future__ import annotations

import csv
import logging
import random
import re
import sys
import time
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Iterator
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------

BASE_URL = "https://www.eldiario.es"
PERIODICO = "eldiario.es"
SITEMAP_URL_TPL = "https://www.eldiario.es/sitemap_contents_{year}_{month:02d}_25b87_001.xml"

OUTPUT_CSV = Path("eldiario_opinion.csv")
SEEN_URLS_FILE = Path("eldiario_seen_urls.txt")

TARGET_ARTICLES = 2500
START_YEAR = 2026
START_MONTH = 5     # arrancamos por el mes en curso e iteramos hacia atrás
MIN_YEAR = 2013     # eldiario.es se fundó en 2012, no buscamos antes
REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_REQS = (0.8, 1.6)
RETRIES = 3
CHECKPOINT_EVERY = 25

# Prefijos de URL que consideramos "opinión".
# Basado en el análisis del sitemap: opinión viene en /opinion/* y /contracorriente/*
# además de blogs de opinión específicos en ediciones locales.
OPINION_PATH_PATTERNS = [
    re.compile(r"^/opinion/"),
    re.compile(r"^/contracorriente/"),
    re.compile(r"^/piedrasdepapel/"),
    re.compile(r"^/contrapoder/"),
    re.compile(r"^/cienciacritica/"),
    re.compile(r"^/caballodenietzsche/"),
    re.compile(r"^/ultima-llamada/"),
    re.compile(r"^/arsenioescolar/"),
    re.compile(r"^/retrones/"),
    # blogs de opinión en ediciones autonómicas
    re.compile(r"^/[^/]+/desdeelsur/"),
    re.compile(r"^/[^/]+/blog/opinion/"),
    re.compile(r"^/[^/]+/lacontradejaen/"),
    re.compile(r"^/[^/]+/canarias-opina/"),
    re.compile(r"^/[^/]+/murcia-y-aparte/"),
    re.compile(r"^/[^/]+/blogs/piedra-de-toque/"),
    re.compile(r"^/madrid/somos/.*opinion"),
    re.compile(r"^/aragon/el-prismatico/"),
]

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("eldiario")

session = requests.Session()
session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept-Language": "es-ES,es;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }
)


def polite_sleep() -> None:
    time.sleep(random.uniform(*SLEEP_BETWEEN_REQS))


def fetch(url: str) -> str | None:
    for attempt in range(1, RETRIES + 1):
        try:
            r = session.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                return r.text
            if r.status_code in (404, 410):
                return None
            log.warning("HTTP %s en %s (intento %d)", r.status_code, url, attempt)
        except requests.RequestException as e:
            log.warning("Error de red en %s: %s (intento %d)", url, e, attempt)
        time.sleep(2 ** attempt)
    return None


# ---------------------------------------------------------------------------
# Sitemaps
# ---------------------------------------------------------------------------

# Regex tolerante: extrae <loc>...</loc> y <lastmod>...</lastmod> de cada <url>.
URL_BLOCK_RE = re.compile(
    r"<url\b[^>]*>(.*?)</url>",
    re.DOTALL,
)
LOC_RE = re.compile(r"<loc>([^<]+)</loc>")
LASTMOD_RE = re.compile(r"<lastmod>([^<]+)</lastmod>")


def is_opinion_url(url: str) -> bool:
    path = urlparse(url).path
    return any(p.search(path) for p in OPINION_PATH_PATTERNS)


def parse_sitemap_for_opinion(xml: str) -> list[tuple[str, str]]:
    """Devuelve [(url, lastmod), ...] solo para URLs de opinión."""
    out: list[tuple[str, str]] = []
    for m in URL_BLOCK_RE.finditer(xml):
        block = m.group(1)
        loc_m = LOC_RE.search(block)
        if not loc_m:
            continue
        url = loc_m.group(1).strip()
        if not is_opinion_url(url):
            continue
        lastmod_m = LASTMOD_RE.search(block)
        lastmod = lastmod_m.group(1).strip() if lastmod_m else ""
        out.append((url, lastmod))
    return out


def iter_months_backwards(start_year: int, start_month: int,
                          min_year: int) -> Iterator[tuple[int, int]]:
    y, m = start_year, start_month
    while y >= min_year:
        yield y, m
        m -= 1
        if m == 0:
            m = 12
            y -= 1


# ---------------------------------------------------------------------------
# Detección paywall y extracción de artículo
# ---------------------------------------------------------------------------

PAYWALL_HTML_SELECTORS = [
    '[class*="paywall" i]',
    '[class*="premium" i]',
    '[id*="paywall" i]',
]
PAYWALL_TEXT_MARKERS = (
    "exclusivo para socios",
    "contenido exclusivo para socios",
    "este artículo es exclusivo",
    "para seguir leyendo, hazte socio",
    "para seguir leyendo, hazte socia",
)


def is_paywalled(soup: BeautifulSoup) -> bool:
    for sel in PAYWALL_HTML_SELECTORS:
        if soup.select_one(sel):
            return True
    article = soup.find("article") or soup.find("main") or soup
    text_low = article.get_text(" ", strip=True).lower()[:6000]
    return any(m in text_low for m in PAYWALL_TEXT_MARKERS)


@dataclass
class Article:
    periodico: str
    titulo: str
    texto: str
    url: str


def extract_article(url: str, html: str) -> Article | None:
    soup = BeautifulSoup(html, "lxml")
    if is_paywalled(soup):
        return None

    h1 = soup.find("h1")
    if not h1:
        return None
    titulo = h1.get_text(" ", strip=True)
    if not titulo:
        return None

    container = soup.find("article") or soup.find("main") or soup
    for sel in [
        "header", "footer", "nav", "aside",
        ".related", ".relacionadas", ".newsletter",
        ".tags", ".share", ".comments", "[class*='comment']",
        "script", "style", "noscript",
    ]:
        for tag in container.select(sel):
            tag.decompose()

    paragraphs: list[str] = []
    for p in container.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt or len(txt) < 25:
            continue
        low = txt.lower()
        if any(k in low for k in ("hazte socio", "hazte socia", "suscríbete a la newsletter")):
            continue
        paragraphs.append(txt)

    texto = "\n\n".join(paragraphs).strip()
    if len(texto) < 150:
        return None

    return Article(periodico=PERIODICO, titulo=titulo, texto=texto, url=url)


# ---------------------------------------------------------------------------
# Persistencia
# ---------------------------------------------------------------------------

CSV_FIELDS = ["periodico", "titulo", "texto", "url"]


def load_seen_urls() -> set[str]:
    if not SEEN_URLS_FILE.exists():
        return set()
    return {line.strip() for line in SEEN_URLS_FILE.read_text(encoding="utf-8").splitlines() if line.strip()}


def append_seen_urls(urls):
    with SEEN_URLS_FILE.open("a", encoding="utf-8") as f:
        for u in urls:
            f.write(u + "\n")


def append_articles_csv(articles: list[Article]):
    write_header = not OUTPUT_CSV.exists()
    with OUTPUT_CSV.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS, quoting=csv.QUOTE_ALL)
        if write_header:
            writer.writeheader()
        for art in articles:
            writer.writerow({
                "periodico": art.periodico,
                "titulo": art.titulo,
                "texto": art.texto,
                "url": art.url,
            })


def count_existing_rows() -> int:
    if not OUTPUT_CSV.exists():
        return 0
    with OUTPUT_CSV.open("r", encoding="utf-8") as f:
        return max(sum(1 for _ in f) - 1, 0)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> int:
    seen_urls = load_seen_urls()
    saved_count = count_existing_rows()
    log.info("Estado inicial: %d artículos en CSV, %d URLs vistas",
             saved_count, len(seen_urls))

    buffer: list[Article] = []
    buffer_seen: list[str] = []
    skipped_paywall = 0
    skipped_error = 0
    months_with_no_sitemap = 0

    pbar = tqdm(total=TARGET_ARTICLES, initial=saved_count, desc="Artículos")

    try:
        for year, month in iter_months_backwards(START_YEAR, START_MONTH, MIN_YEAR):
            if saved_count >= TARGET_ARTICLES:
                break

            sm_url = SITEMAP_URL_TPL.format(year=year, month=month)
            log.info("=== Sitemap %04d-%02d ===  %s", year, month, sm_url)
            xml = fetch(sm_url)
            polite_sleep()
            if not xml:
                months_with_no_sitemap += 1
                log.warning("Sin sitemap para %04d-%02d (404 u otro error)", year, month)
                if months_with_no_sitemap >= 3:
                    log.error("3 meses seguidos sin sitemap, paro.")
                    break
                continue
            months_with_no_sitemap = 0

            entries = parse_sitemap_for_opinion(xml)
            log.info("  Mes %04d-%02d: %d URLs de opinión", year, month, len(entries))

            # Procesamos en orden cronológico inverso dentro del mes
            entries.sort(key=lambda e: e[1], reverse=True)

            for url, lastmod in entries:
                if saved_count >= TARGET_ARTICLES:
                    break
                if url in seen_urls:
                    continue

                html = fetch(url)
                polite_sleep()
                seen_urls.add(url)
                buffer_seen.append(url)

                if not html:
                    skipped_error += 1
                    continue

                art = extract_article(url, html)
                if art is None:
                    skipped_paywall += 1
                    continue

                buffer.append(art)
                saved_count += 1
                pbar.update(1)

                if len(buffer) >= CHECKPOINT_EVERY:
                    append_articles_csv(buffer)
                    append_seen_urls(buffer_seen)
                    log.info("Checkpoint +%d (total %d). Paywall: %d. Errores: %d",
                             len(buffer), saved_count, skipped_paywall, skipped_error)
                    buffer.clear()
                    buffer_seen.clear()

    except KeyboardInterrupt:
        log.warning("Interrumpido por usuario")

    finally:
        if buffer:
            append_articles_csv(buffer)
            append_seen_urls(buffer_seen)
        pbar.close()
        log.info("FIN. Guardados: %d. Paywall: %d. Errores: %d.",
                 saved_count, skipped_paywall, skipped_error)
    return 0


if __name__ == "__main__":
    sys.exit(main())

11:38:55 [INFO] Estado inicial: 0 artículos en CSV, 40 URLs vistas
Artículos:   0%|          | 0/2500 [00:00<?, ?it/s]11:38:55 [INFO] === Sitemap 2026-05 ===  https://www.eldiario.es/sitemap_contents_2026_05_25b87_001.xml
11:38:56 [INFO]   Mes 2026-05: 17 URLs de opinión
Artículos:   0%|          | 4/2500 [00:10<1:53:32,  2.73s/it]11:39:05 [INFO] === Sitemap 2026-04 ===  https://www.eldiario.es/sitemap_contents_2026_04_25b87_001.xml
11:39:07 [INFO]   Mes 2026-04: 302 URLs de opinión
Artículos:   4%|▍         | 100/2500 [08:41<3:00:27,  4.51s/it]11:47:36 [INFO] Checkpoint +25 (total 100). Paywall: 175. Errores: 0
11:47:36 [INFO] === Sitemap 2026-03 ===  https://www.eldiario.es/sitemap_contents_2026_03_25b87_001.xml
11:47:39 [INFO]   Mes 2026-03: 319 URLs de opinión
Artículos:   8%|▊         | 209/2500 [20:37<3:44:31,  5.88s/it]11:59:33 [INFO] === Sitemap 2026-02 ===  https://www.eldiario.es/sitemap_contents_2026_02_25b87_001.xml
11:59:35 [INFO]   Mes 2026-02: 315 URLs de opinión
Artícul

SystemExit: 0